In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import cv2

In [2]:
IMG_SIZE = 224
NUM_FRAMES = 15
BATCH_SIZE = 4
NUM_CLASSES = 1
LEARNING_RATE = 1e-4

In [3]:
import os

def count_min_frames(data_dir):
    min_frames = float('inf')
    min_video_path = None

    for class_name in ['Celeb-real', 'Celeb-synthesis']:
        class_path = os.path.join(data_dir, class_name)

        for video_folder in os.listdir(class_path):
            video_path = os.path.join(class_path, video_folder)

            if not os.path.isdir(video_path):
                continue

            frame_files = [
                f for f in os.listdir(video_path)
                if f.endswith(('.jpg', '.png'))
            ]

            num_frames = len(frame_files)

            if num_frames < min_frames:
                min_frames = num_frames
                min_video_path = video_path

    return min_frames

In [4]:
NUM_FRAMES =  count_min_frames("C:\\Users\\RGUKT\\Desktop\\deepfake\\models\\dataset_faces\\train")
NUM_FRAMES

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\RGUKT\\Desktop\\deepfake\\models\\dataset_faces\\train\\Celeb-real'

In [ ]:
def load_video_frames(video_folder):
    frames = sorted(os.listdir(video_folder))[:NUM_FRAMES]
    images = []

    for frame in frames:
        img_path = os.path.join(video_folder, frame)
        img = tf.keras.preprocessing.image.load_img(
            img_path, target_size=(IMG_SIZE, IMG_SIZE)
        )
        img = tf.keras.preprocessing.image.img_to_array(img) / 255.0
        images.append(img)

    return tf.stack(images)

In [ ]:
def create_dataset(data_dir):
    videos = []
    labels = []

    for label_name in ['Celeb-real', 'Celeb-synthesis']:
        label_path = os.path.join(data_dir, label_name)

        for video_folder in os.listdir(label_path):
            video_path = os.path.join(label_path, video_folder)
            videos.append(load_video_frames(video_path))
            labels.append(0 if label_name == 'Celeb-real' else 1)

    return tf.stack(videos), tf.convert_to_tensor(labels)

In [ ]:
X_train, y_train = create_dataset("C:\\Users\\RGUKT\\Desktop\\deepfake\\models\\dataset_faces\\train")
X_val, y_val = create_dataset("C:\\Users\\RGUKT\\Desktop\\deepfake\\models\\dataset_faces\\val")

KeyboardInterrupt: 

In [ ]:
import os

def count_min_frames(data_dir):
    min_frames = float('inf')
    min_video_path = None

    for class_name in ['Celeb-real', 'Celeb-synthesis']:
        class_path = os.path.join(data_dir, class_name)

        for video_folder in os.listdir(class_path):
            video_path = os.path.join(class_path, video_folder)

            if not os.path.isdir(video_path):
                continue

            frame_files = [
                f for f in os.listdir(video_path)
                if f.endswith(('.jpg', '.png'))
            ]

            num_frames = len(frame_files)

            if num_frames < min_frames:
                min_frames = num_frames
                min_video_path = video_path

    return min_frames

In [ ]:
import tensorflow as tf
import os
import cv2
import numpy as np

IMG_SIZE = 224
NUM_FRAMES =  count_min_frames("C:\\Users\\RGUKT\\Desktop\\deepfake\\models\\dataset_faces\\train")
BATCH_SIZE = 4

def load_video_frames(video_path):
    frame_files = sorted([
        f for f in os.listdir(video_path)
        if f.endswith(('.jpg', '.png'))
    ])

    frame_files = frame_files[:NUM_FRAMES]

    frames = []

    for frame in frame_files:
        img_path = os.path.join(video_path, frame)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        frames.append(img)

    # if len(frames) < NUM_FRAMES:
    #     return None

    return np.array(frames, dtype=np.float32)


def generator(data_dir):

    for label_name in ['Celeb-real', 'Celeb-synthesis']:
        class_path = os.path.join(data_dir, label_name)
        label = 0 if label_name == 'Celeb-real' else 1

        for video in os.listdir(class_path):
            video_path = os.path.join(class_path, video)

            frames = load_video_frames(video_path)

            if frames is None:
                continue

            yield frames, label

In [ ]:
output_signature = (
    tf.TensorSpec(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: generator("C:\\Users\\RGUKT\\Desktop\\deepfake\\models\\dataset_faces\\train"),
    output_signature=output_signature
)

train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
train_dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 17, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [ ]:
output_signature = (
    tf.TensorSpec(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: generator("dataset/train"),
    output_signature=output_signature
)

train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)